# 📊 01. Exploratory Data Analysis & Collection History
### PreZero / Fillinq — Zakaria’s glass-routing pipeline

We compare recorded glass streams, calendar patterns and actual pre-emptying visit states. Every row is a collection observation. `overfull` remains a category, not an invented percentage. The original exploratory structure is retained, with the observation interpretation corrected.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "Zakaria-Work"
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from Zakaria-Work or its parent directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loader import load_and_clean_data, collection_summary
from src.clustering import extract_container_profiles, cluster_containers
from src.predictor import predict_container_demands, select_collection_sites
OUT = ROOT / "data" / "corrected"
OUT.mkdir(parents=True, exist_ok=True)
MAT_COLORS = {"white": "#1f77b4", "green": "#2ca02c", "brown": "#8c564b"}
plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 100})


## 1. Data ingestion and overview
The loader validates identities, dates and fill values. It retains raw fill text and a separate overflow flag. Invalid or duplicate readings are rejected explicitly; source files are read-only.

In [ ]:
df = load_and_clean_data(ROOT / "glass_fill_levels_anonymized_f.csv")
print(f"{len(df):,} pre-emptying observations; {df.Location_id.nunique()} locations; "
      f"{len(df.groupby(['Location_id', 'material_type']))} observed bins")
print(f"Coverage: {df.date.min().date()} to {df.date.max().date()}")
display(df.head())


---
## 📈 2. Dataset Health Check & Core KPIs


In [ ]:
summary_df = pd.DataFrame({'Metric':['Readings','Locations','Observed location/material pairs','First date','Last date','Overfull observations','Numeric readings'],
    'Value':[len(df),df.Location_id.nunique(),len(df.groupby(['Location_id','material_type'])),str(df.date.min().date()),str(df.date.max().date()),int(df.is_overfull.sum()),int(df.fill_num.notna().sum())]})
display(summary_df)


## 3. White, green and brown glass
Compare fill at visits and the explicit overflow category. Readings are sparse and visit-driven; they are not simultaneous network occupancy.

In [ ]:
stream_comparison_df = collection_summary(df)
stream_comparison_df['overfull_share_pct'] = (100 * stream_comparison_df.overfull_count / stream_comparison_df.collections)
display(stream_comparison_df.round(2))
print("Means exclude overfull. Overfull shares use every collection observation as denominator.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Fill Level Distribution
fl_counts = df['fill_level_perc'].value_counts()
ordered_cats = ['0', '10', '20', '30', '40', '50', '60', '70', '80', '90', '100', 'overfull']
fl_ordered = fl_counts.reindex(ordered_cats).fillna(0)
bar_colors = ['#2ca02c' if c in ['10', '20', '30', '40'] else '#ff7f0e' if c in ['50', '60', '70', '80'] else '#d62728' for c in ordered_cats]

sns.barplot(x=fl_ordered.index, y=fl_ordered.values, ax=axes[0], palette=bar_colors)
axes[0].set_title("Overall Fill Level Distribution", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Recorded Fill Level (%)")
axes[0].set_ylabel("Count")

# 2. Material Breakdown
mat_counts = df['material_type'].value_counts()
sns.barplot(x=mat_counts.index, y=mat_counts.values, ax=axes[1], palette=[MAT_COLORS[m] for m in mat_counts.index])
axes[1].set_title("Observations by Glass Stream", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Material Type")
axes[1].set_ylabel("Count")
for i, (m, v) in enumerate(mat_counts.items()):
    axes[1].text(i, v + 40, f"{v:,} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Calendar dynamics
Separate actual BW public holidays from the broader Christmas/New Year window. These comparisons describe collection observations, not a measured holiday-caused change in demand.

In [ ]:
dow = df.groupby('day_name').agg(readings=('date','size'), numeric_mean=('fill_num','mean')).reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
monthly = df.pivot_table(index='year_month', columns='material_type', values='fill_num', aggfunc='mean')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
dow.readings.plot.bar(ax=axes[0], color='#4676a3')
axes[0].set(title='Recorded collection activity by weekday', ylabel='Observations')
monthly.plot(ax=axes[1], marker='o', color=[MAT_COLORS[m] for m in monthly.columns])
axes[1].set(title='Monthly numeric pre-emptying fill', ylabel='Numeric mean (%)', xlabel='Month')
plt.tight_layout()
plt.show()
display(df.groupby('is_public_holiday').agg(readings=('date','size'), numeric_mean=('fill_num','mean'), overfull_share=('is_overfull','mean')))
print("Calendar comparisons are descriptive: visit timing and coverage can confound them.")


## 5. Collection-state audit
Every recorded visit precedes emptying. Count the observed states directly; a drop between visits does not identify an emptying, and a positive difference is not within-cycle growth. The old late/premature efficiency percentages are withdrawn.

In [ ]:
audit = pd.DataFrame({
    'Recorded state': ['Numeric 0–30%', 'Numeric above 30/below 70%', 'Numeric 70/below 100%', 'Numeric 100%', 'Explicit overfull'],
    'Collections': [df.fill_num.le(30).sum(), (df.fill_num.gt(30)&df.fill_num.lt(70)).sum(),
                    (df.fill_num.ge(70)&df.fill_num.lt(100)).sum(), df.fill_num.eq(100).sum(), df.is_overfull.sum()]
})
audit['Share of collections (%)'] = audit.Collections / len(df) * 100
display(audit.round(2))
ax = audit.plot.barh(x='Recorded state', y='Collections', legend=False, color='#4676a3')
ax.set_title('What was observed immediately before each recorded emptying')
plt.tight_layout()
plt.show()
print("These bands describe visits; they do not prove a collection was avoidable.")


## 6. Export corrected data
Corrected outputs live in `data/corrected/`. Earlier CSVs in `data/` are legacy exports and are not reused by the corrected notebooks.

In [ ]:
df.to_csv(OUT / 'cleaned_fill_levels.csv', index=False)
stream_comparison_df.to_csv(OUT / 'collection_summary.csv', index=False)
print(OUT)
